# Step 2 — Append only the new experimental day

**Incremental workflow for an existing project; historical data are not recomputed.** Each mouse's reference comes from the existing `metadata_registration.csv`. If the original Step2 selected a fallback reference rather than Day8, this procedure preserves the actually selected reference. The source modules, YAML, and historical results remain unchanged.

Confirm that the new day is already present in `manifest_recordings.csv` and that reference BMP and Ilastik probability H5 paths are populated.

**Do not run the original notebook's full `run_step2()` workflow.**


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

SRC_DIR = Path(r"...\src")
CONFIG_PATH = Path(r"config.yaml")  # Match the configuration used for the existing Step1 project.
NEW_EXP_DAY = "PT"  # Exact exp_day value in the manifest, e.g., 7d-PT.

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from step2_add_exp_day import plan_add_step2_day, commit_add_step2_day

assert CONFIG_PATH.is_file(), f"Config not found: {CONFIG_PATH}"
assert NEW_EXP_DAY != "NEW_EXP_DAY", "Укажите NEW_EXP_DAY"


## 1. Preflight — do not write into historical Step2 outputs

Inspect the number of candidate recordings and the **actual stored `ref_stem`** for each mouse. Day8 is not assumed. Resolve missing H5 files, reference images, or structural inconsistencies before committing.


In [ ]:
plan = plan_add_step2_day(CONFIG_PATH, NEW_EXP_DAY)
print(json.dumps(plan.summary(), indent=2, ensure_ascii=False))
cols = [c for c in ["stem", "group_id", "mouse_id", "exp_day", "state", "protocol",
                    "ref_505_path", "prob_505_path", "ref_656_path", "prob_656_path"] if c in plan.candidates]
display(plan.candidates[cols])
if plan.collisions:
    print("СТОП: обнаружены уже существующие файлы новых stem", plan.collisions)


## 2. Explicit commit

**Warning: the current code cell has `COMMIT = True`.** Execution can write new Step2 artifacts and append metadata. Change it to `False` for review; set `True` only after checking the candidate stems and historical references.

If the manifest, YAML, or `metadata_registration.csv` changes after preflight, commit aborts.


In [ ]:
COMMIT = True  # Switch to True only after reviewing plan.summary().
if COMMIT:
    result = commit_add_step2_day(plan)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("Запись не выполнялась. Для добавления новых stem установите COMMIT = True.")


## 3. Validate the new Step2 rows

After commit, verify `ref_stem`, bregma and QC flags for the newly registered records. For `bregma_empty` or `is_bad=True`, **do not rerun the full Step2**; use a targeted, safe manual correction workflow for the affected stems.


In [ ]:
if COMMIT:
    meta = pd.read_csv(plan.paths["metadata_registration_csv"])
    selected = meta[meta["stem"].astype(str).isin(plan.candidates["stem"].astype(str))]
    qc_cols = [c for c in ["mouse_id", "exp_day", "stem", "group_id", "ref_stem", "ref_exact_match",
                             "bregma_found", "bregma_cy", "bregma_cx", "is_bad",
                             "bad_reasons", "score", "saved_registered_path"] if c in selected]
    display(selected[qc_cols])
    print(f"Добавлено строк: {len(selected)}")


In [ ]:
import pandas as pd

metadata_path = (
    r"H:\PT2021_final_analysis"
    r"\Main_analysis_2026-10-15"
    r"\out\step2_masks\metadata_registration.csv"
)

df = pd.read_csv(metadata_path)
pt = df[df["exp_day"].astype(str).eq("PT")].copy()

print("Количество записей PT:", len(pt))

qc_cols = [
    c for c in pt.columns
    if any(x in c.lower() for x in
           ["status", "is_bad", "bregma", "error"])
]

print(pt[["stem", "ref_stem"] + qc_cols].to_string(index=False))